# House Price Prediction - Exploratory Data Analysis and Model Benchmarking
This notebook demonstrates the end-to-end machine learning pipeline for real estate valuation using 50,000 verified property records.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from xgboost import XGBRegressor
import os

## 1. Load Dataset
Load the 50,000 verified property transactions dataset.

In [ ]:
data_path = "../data/housing_data.csv" if os.path.exists("../data/housing_data.csv") else "data/housing_data.csv"
df = pd.read_csv(data_path)
print(f"Dataset Shape: {df.shape}")
df.head()

## 2. Statistical Summary

In [ ]:
df.describe()

## 3. Data Preprocessing
Encode categorical columns and prepare feature matrices.

In [ ]:
categorical_cols = ["Location", "Furnishing", "Road_access", "Guestroom", "Basement", "Hot_water", "AC", "Preferred_area"]
df_processed = df.copy()
for col in categorical_cols:
    df_processed[col] = LabelEncoder().fit_transform(df_processed[col])

X = df_processed.drop(columns=["Price"])
y = df_processed["Price"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")

## 4. Model Training & Benchmarking
Train 6 candidate regression models and evaluate performance on holdout test set.

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Lasso Regression": Lasso(alpha=1.0),
    "Random Forest": RandomForestRegressor(n_estimators=50, max_depth=16, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=100, max_depth=6, random_state=42, n_jobs=-1, verbosity=0)
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    r2 = r2_score(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    mae = mean_absolute_error(y_test, pred)
    results.append({"Model": name, "R2_Score": round(r2, 4), "RMSE": round(rmse, 2), "MAE": round(mae, 2)})

results_df = pd.DataFrame(results).sort_values(by="R2_Score", ascending=False)
results_df